In [1]:
import pandas as pd
import nltk
import re
from nltk.stem import WordNetLemmatizer

In [2]:
file_name = '../data/ACLED Data_2026-06-10.csv'

df = pd.read_csv(file_name, delimiter=',')
print(df.columns)
print(df)
df['tags'].unique()

Index(['event_id_cnty', 'event_date', 'year', 'time_precision',
       'disorder_type', 'event_type', 'sub_event_type', 'actor1',
       'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2',
       'interaction', 'civilian_targeting', 'iso', 'region', 'country',
       'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude',
       'geo_precision', 'source', 'source_scale', 'notes', 'fatalities',
       'tags', 'timestamp', 'population_best'],
      dtype='str')
       event_id_cnty  event_date  year  time_precision       disorder_type  \
0              UKR22  2018-01-01  2018               1  Political violence   
1              UKR24  2018-01-01  2018               1  Political violence   
2              UKR27  2018-01-01  2018               1  Political violence   
3              UKR21  2018-01-01  2018               1  Political violence   
4              UKR30  2018-01-01  2018               1  Political violence   
...              ...         ...   ...       

<ArrowStringArray>
[                                                   nan,
                             'crowd size=more than 100',
                             'crowd size=more than 200',
                                 'crowd size=about 200',
                                 'crowd size=no report',
                                'crowd size=about 1000',
                                 'crowd size=about 100',
                                  'crowd size=about 10',
                               'crowd size=a few dozen',
                             'crowd size=several dozen',
 ...
                                 'crowd size=2500-6000',
                          'crowd size=no more than 6-7',
                'crowd size=more than several hundreds',
                      'crowd size=dozens-more than 200',
 'crowd size=between several thousand to around 10,000',
            'crowd size=between around 2,700 and 5,000',
                                'crowd size=around 445',
       

In [3]:

df['event_type'].unique()
df = df.drop_duplicates(subset=['notes', 'event_date', 'country'])
print(df)

       event_id_cnty  event_date  year  time_precision       disorder_type  \
0              UKR22  2018-01-01  2018               1  Political violence   
1              UKR24  2018-01-01  2018               1  Political violence   
2              UKR27  2018-01-01  2018               1  Political violence   
3              UKR21  2018-01-01  2018               1  Political violence   
4              UKR30  2018-01-01  2018               1  Political violence   
...              ...         ...   ...             ...                 ...   
616968     UKR325474  2026-06-01  2026               1  Political violence   
616969     UKR325689  2026-06-01  2026               1  Political violence   
616970     UKR325742  2026-06-01  2026               1  Political violence   
616971     UKR325743  2026-06-01  2026               1  Political violence   
616972     UKR325859  2026-06-01  2026               1  Political violence   

                        event_type                     sub_even

In [4]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
print(stopwords.words('english')[:10])
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = re.sub(r'\W+', ' ', str(text).lower())
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)
regex_pattern =  r"^(?:\[)?.?\d{4}(?:[,.:]?\s)?"

df.notes = df['notes'].str.replace(regex_pattern, '', regex=True).str.strip()
df['clean_notes'] = df['notes'].apply(preprocess)
df['clean_notes'] = df['clean_notes'].apply(lambda x: ' '.join(str(x).split()[3:]))
df = df[df['clean_notes'].str.strip() != '']

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


[nltk_data] Downloading package wordnet to /home/martan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /home/martan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
ndf = df[df['event_type'].isin(['Protests', 'Riots'])] #maybe 'Violence against civilians'
ndf = ndf[ndf['sub_event_type'] != 'Mob violence']
print(ndf)

       event_id_cnty  event_date  year  time_precision   disorder_type  \
12          UKR39144  2018-01-01  2018               1  Demonstrations   
13          UKR39145  2018-01-01  2018               1  Demonstrations   
14          UKR39143  2018-01-01  2018               1  Demonstrations   
19              UKR7  2018-01-01  2018               1  Demonstrations   
20              UKR4  2018-01-01  2018               1  Demonstrations   
...              ...         ...   ...             ...             ...   
616963       BEL5441  2026-06-01  2026               1  Demonstrations   
616964       BEL5442  2026-06-01  2026               1  Demonstrations   
616965       NLD5811  2026-06-01  2026               1  Demonstrations   
616966       POR3010  2026-06-01  2026               1  Demonstrations   
616967      RUS59638  2026-06-01  2026               1  Demonstrations   

       event_type    sub_event_type                    actor1  \
12       Protests  Peaceful protest      Prote

In [6]:
ndf['country_code'] = ndf['event_id_cnty'].str[:3]
print(ndf['country_code'].unique())
european_codes = [
    'ALB', 'AND', 'AUT', 'BEL', 'BGR', 'BIH', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK',
    'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GRC', 'HRV', 'HUN', 'IRL', 'ISL', 'ITA',
    'KOS', 'LIE', 'LTU', 'LUX', 'LVA', 'MCO', 'MDA', 'MKD', 'MLT', 'MNE', 'NLD',
    'NOR', 'POL', 'PRT', 'ROU', 'SMR', 'SRB', 'SVK', 'SVN', 'SWE', 'VAT'
]

nnndf = ndf[ndf['country_code'].isin(european_codes)]
print(nnndf)


european_countries = [
    'Albania', 'Andorra', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria',
    'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France',
    'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Kosovo',
    'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova',
    'Monaco', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland',
    'Portugal', 'Romania', 'San Marino', 'Serbia', 'Slovakia', 'Slovenia',
    'Spain', 'Sweden', 'Switzerland', 'United Kingdom', 'Vatican City'
]

nndf = ndf[ndf['country'].isin(european_countries)]
print(nndf)

<ArrowStringArray>
['UKR', 'GRC', 'BGR', 'BLR', 'RUS', 'XKX', 'SRB', 'BIH', 'ROU', 'HRV', 'MDA',
 'ALB', 'MNE', 'MKD', 'CYP', 'XSB', 'GBR', 'DEU', 'IRL', 'ESP', 'ITA', 'NLD',
 'NOR', 'FRA', 'POL', 'BEL', 'LTU', 'DNK', 'AUT', 'EST', 'SWE', 'POR', 'MLT',
 'HUN', 'CHE', 'SVN', 'FIN', 'CZE', 'LUX', 'SVK', 'LVA', 'AND', 'IMN', 'GGY',
 'ISL', 'JEY', 'GRL', 'FRO', 'GIB', 'VAT', 'LIE', 'SMR', 'MCO']
Length: 53, dtype: str
       event_id_cnty  event_date  year  time_precision   disorder_type  \
49           GRC2057  2018-01-02  2018               1  Demonstrations   
75              BGR1  2018-01-03  2018               1  Demonstrations   
76           GRC2038  2018-01-03  2018               1  Demonstrations   
83             BGR10  2018-01-04  2018               1  Demonstrations   
84             BGR11  2018-01-04  2018               1  Demonstrations   
...              ...         ...   ...             ...             ...   
616961      FRA46910  2026-06-01  2026               1  Demonstr

In [7]:
nndf.to_csv('../data/filtered_events.csv', index=False)
nnndf.to_csv('../data/filtered_events_country_code.csv', index=False)

In [8]:
nnndf['sub_event_type'].unique()

<ArrowStringArray>
[                  'Peaceful protest',          'Protest with intervention',
              'Violent demonstration', 'Excessive force against protesters']
Length: 4, dtype: str